In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

/kaggle/input/pss5e10-oofs/test_preds_autogluon_baseline.csv
/kaggle/input/pss5e10-oofs/oofs_autogluon_baseline.csv
/kaggle/input/pss5e10-oofs/test_preds_autogluon_residuals.csv
/kaggle/input/pss5e10-oofs/oofs_autogluon_residuals.csv
/kaggle/input/s5e10-realmlp-tuned/oof_realmlp_plus_origcol.csv
/kaggle/input/s5e10-realmlp-tuned/test_realmlp_plus_origcol.csv
/kaggle/input/s5e10-realmlp-tuned/__results__.html
/kaggle/input/s5e10-realmlp-tuned/__notebook__.ipynb
/kaggle/input/s5e10-realmlp-tuned/__output__.json
/kaggle/input/s5e10-realmlp-tuned/custom.css
/kaggle/input/s5e10-tabm-over-residuals/oof_tabm_overresid.csv
/kaggle/input/s5e10-tabm-over-residuals/__results__.html
/kaggle/input/s5e10-tabm-over-residuals/__notebook__.ipynb
/kaggle/input/s5e10-tabm-over-residuals/__output__.json
/kaggle/input/s5e10-tabm-over-residuals/test_tabm_overresid.csv
/kaggle/input/s5e10-tabm-over-residuals/custom.css
/kaggle/input/s5e10-lgbm-origcol-20seeds/oof_20seedslgb_plus_origcol.csv
/kaggle/input/s5e

In [2]:
!pip install autogluon.tabular[0]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 225.1/225.1 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.2/64.2 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.0/71.0 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 31.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 487.3/487.3 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.0/278.0 kB 12.1 MB/s eta 0:00:00
  Attempting uninstall: psutil
    Found existing installation: psutil 7.1.0
    Uninstalling psutil-7.1.0:
      Successfully uninstalled psutil-7.1.0
  Attempting uninstall: scikit-learn
    Found existing installation: scikit-learn 1.2.2
    Uninstalling scikit-learn-1.2.2:
      Successfully uninstalled scikit-learn-1.2.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
category-encoders 2.7.

In [3]:
# train = pd.read_csv('/kaggle/input/pss5e10-main/final_train.csv')
# # train = train.fillna(0)
# test = pd.read_csv('/kaggle/input/pss5e10-main/test.csv')
# test = test.drop(columns='accident_risk')
# train['residual_risk'] = train['accident_risk'] - train['y']
# train.drop(columns='accident_risk', inplace=True)

In [4]:
train_final = pd.read_csv('/kaggle/input/pss5e10-main/final_train.csv')
test_final = pd.read_csv('/kaggle/input/pss5e10-main/test.csv')
train_final.drop(columns='accident_risk', inplace=True)
test_final.drop(columns='accident_risk', inplace=True)

oofs_df_baseline = pd.read_csv('/kaggle/input/pss5e10-oofs/oofs_autogluon_baseline.csv')
test_df_baseline = pd.read_csv('/kaggle/input/pss5e10-oofs/test_preds_autogluon_baseline.csv')
oofs_df_residuals = pd.read_csv('/kaggle/input/pss5e10-oofs/oofs_autogluon_residuals.csv')
test_df_residuals = pd.read_csv('/kaggle/input/pss5e10-oofs/test_preds_autogluon_residuals.csv')

oofs_df_residuals.columns = [col + '_res' for col in oofs_df_residuals.columns]
test_df_residuals.columns = [col + '_res' for col in test_df_residuals.columns]

oofs_df_tabm = pd.read_csv('/kaggle/input/s5e10-single-tabm-tuned/oof_tabm_plus_origcol_tuned.csv')
test_df_tabm = pd.read_csv('/kaggle/input/s5e10-single-tabm-tuned/test_tabm_plus_origcol_tuned.csv')

oofs_df_lgbm = pd.read_csv('/kaggle/input/s5e10-lgbm-origcol-20seeds/oof_20seedslgb_plus_origcol.csv')
test_df_lgbm = pd.read_csv('/kaggle/input/s5e10-lgbm-origcol-20seeds/test_20seedslgb_plus_origcol.csv')

oofs_df_xgb = pd.read_csv('/kaggle/input/s5e10-xgb-origcol-20seeds/oof_xgb_plus_origcol.csv')
test_df_xgb = pd.read_csv('/kaggle/input/s5e10-xgb-origcol-20seeds/test_xgb_plus_origcol.csv')

oofs_df_mlp = pd.read_csv('/kaggle/input/s5e10-realmlp-tuned/oof_realmlp_plus_origcol.csv')
test_df_mlp = pd.read_csv('/kaggle/input/s5e10-realmlp-tuned/test_realmlp_plus_origcol.csv')

oofs_df_tabm_res = pd.read_csv('/kaggle/input/s5e10-tabm-over-residuals/oof_tabm_overresid.csv')
test_df_tabm_res = pd.read_csv('/kaggle/input/s5e10-tabm-over-residuals/test_tabm_overresid.csv')

oofs_df_xgb2 = pd.read_csv('/kaggle/input/pss5e10-main/xgb_20seed_oof_residuals.csv')
test_df_xgb2 = pd.read_csv('/kaggle/input/pss5e10-main/xgb_20seed_test_residuals.csv')

oofs_df_cat2 = pd.read_csv('/kaggle/input/pss5e10-main/cat_20seed_oof_residuals.csv')
test_df_cat2 = pd.read_csv('/kaggle/input/pss5e10-main/cat_20seed_test_residuals.csv')

oofs_df_lgb2 = pd.read_csv('/kaggle/input/pss5e10-main/lgb_20seed_oof_residuals.csv')
test_df_lgb2 = pd.read_csv('/kaggle/input/pss5e10-main/lgb_20seed_test_residuals.csv')

oofs_df = pd.concat([
    oofs_df_tabm.drop(columns='id').add_prefix('tabm_'),
    oofs_df_lgbm.drop(columns='id').add_prefix('lgbm_'),
    oofs_df_xgb.drop(columns='id').add_prefix('xgb_'),
    oofs_df_mlp.drop(columns='id').add_prefix('mlp_'),
    oofs_df_tabm_res.drop(columns='id').add_prefix('tabm_res_'),
    # oofs_df_xgb2.drop(columns=['id', 'Unnamed: 0']).add_prefix('xgb2_'),
    oofs_df_cat2.drop(columns=['id', 'Unnamed: 0']).add_prefix('cat2_'),
    oofs_df_lgb2.drop(columns='id').add_prefix('lgb2_'),
    train_final,
], axis=1)

test_df = pd.concat([
    test_df_tabm.drop(columns='id').add_prefix('tabm_'),
    test_df_lgbm.drop(columns='id').add_prefix('lgbm_'),
    test_df_xgb.drop(columns='id').add_prefix('xgb_'),
    test_df_mlp.drop(columns='id').add_prefix('mlp_'),
    test_df_tabm_res.drop(columns='id').add_prefix('tabm_res_'),
    # test_df_xgb2.drop(columns=['id', 'Unnamed: 0']).add_prefix('xgb2_'),
    test_df_cat2.drop(columns=['id', 'Unnamed: 0']).add_prefix('cat2_'),
    test_df_lgb2.drop(columns='id').add_prefix('lgb2_'),
    test_final,
], axis=1)

# oofs_df = pd.concat([oofs_df_baseline, oofs_df_residuals], axis=1)
# test_df = pd.concat([test_df_baseline, test_df_residuals], axis=1)

train = pd.read_csv('/kaggle/input/playground-series-s5e10/train.csv')
y = train['accident_risk']

In [5]:
TARGET = 'accident_risk'
FEATURES = [col for col in oofs_df.columns if col!='accident_risk']

In [6]:
# oofs_df = pd.concat([oofs_df, y], axis=1)
train_final = pd.concat([train_final, y], axis=1)

In [7]:
# train = train.fillna(0)
# test = test.fillna(0)

In [8]:
import shutil, os
old = '/kaggle/working/AutogluonModels'
if os.path.exists(old):
    shutil.rmtree(old)

In [9]:
import autogluon.core.utils.utils as core_utils
from sklearn.model_selection import RepeatedKFold, RepeatedStratifiedKFold, LeaveOneGroupOut

_ORIG_CVSPLITTER_INIT = core_utils.CVSplitter.__init__

def _cvsplitter_init_with_42(self, splitter_cls=None, n_splits=5, n_repeats=1,
                             random_state=None, stratify=False, bin=False,
                             n_bins=None, groups=None):
    # force our seed, ignore the 0 that the trainer passes
    _ORIG_CVSPLITTER_INIT(
        self,
        splitter_cls=splitter_cls,
        n_splits=n_splits,
        n_repeats=n_repeats,
        random_state=42,        # <-- your seed
        stratify=stratify,
        bin=bin,
        n_bins=n_bins,
        groups=groups,
    )

core_utils.CVSplitter.__init__ = _cvsplitter_init_with_42

In [10]:
from autogluon.tabular import TabularPredictor
import warnings
warnings.filterwarnings('ignore')

# PEAK_XGB = {
#     'n_estimators': 100_000, 'learning_rate': 0.01, 'max_depth': 6,
#     'subsample': 0.9, 'colsample_bytree': 0.6, 'reg_alpha': 0.0, 'reg_lambda': 0.0,
#     'tree_method': 'gpu_hist', 'device': 'cuda', 'n_jobs': -1, 'verbosity': 0
# }

# PEAK_LGB = {
#     'n_estimators': 100_000, 'learning_rate': 0.01, 'num_leaves': 64,
#     'subsample': 0.9, 'colsample_bytree': 0.6, 'reg_alpha': 0.0, 'reg_lambda': 0.0,
#     'device': 'gpu', 'n_jobs': -1, 'verbosity': -1
# }

# PEAK_CAT = {
#     'iterations': 100_000, 'learning_rate': 0.01, 'depth': 6,
#     'l2_leaf_reg': 0.0, 'subsample': 0.9, 'task_type': 'GPU', 'verbose': False
# }

# ----------  AutoGluon search space  ----------
predictor = TabularPredictor(
    label=TARGET,
    eval_metric='rmse',
    problem_type='regression',
    path='AutogluonModels/full_hpo'
).fit(
    train_data=train_final,
    time_limit=3600*11,  # 2 hours for extensive HPO
    presets='best_quality',
    num_bag_folds=5,
    num_stack_levels=3,
    num_bag_sets=3,
    auto_stack=True,
    raise_on_no_models_fitted=False,
    # REMOVED: hyperparameter_tune=True,  # Not needed - just use hyperparameter_tune_kwargs
    # hyperparameter_tune_kwargs={
    #     'scheduler': 'local',
    #     'searcher': 'bayesopt',
    #     'num_trials': 50,
    # },
    # hyperparameters={
    #     # XGBoost with search space
    #     'XGB': {
    #         'n_estimators': 10000,
    #         'learning_rate': [0.005, 0.01, 0.02, 0.05, 0.1],
    #         'max_depth': [4, 5, 6, 7, 8, 9],
    #         'subsample': [0.7, 0.8, 0.9, 1.0],
    #         'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    #         'reg_alpha': [0, 0.1, 0.5, 1.0],
    #         'reg_lambda': [0, 0.1, 0.5, 1.0, 5.0],
    #         'min_child_weight': [1, 3, 5, 7],
    #         'tree_method': 'gpu_hist',
    #         'device': 'cuda',
    #     },
        
    #     # LightGBM with search space
    #     'GBM': {
    #         'n_estimators': 10000,
    #         'learning_rate': [0.005, 0.01, 0.02, 0.05, 0.1],
    #         'num_leaves': [31, 63, 127, 255],
    #         'max_depth': [6, 8, 10, 12, -1],
    #         'subsample': [0.7, 0.8, 0.9, 1.0],
    #         'colsample_bytree': [0.6, 0.7, 0.8, 0.9, 1.0],
    #         'reg_alpha': [0, 0.1, 0.5, 1.0],
    #         'reg_lambda': [0, 0.1, 0.5, 1.0, 5.0],
    #         'min_child_samples': [5, 10, 20, 30],
    #         'device': 'gpu',
    #     },
        
    #     # CatBoost with search space
    #     'CAT': {
    #         'iterations': 10000,
    #         'learning_rate': [0.005, 0.01, 0.02, 0.05, 0.1],
    #         'depth': [4, 5, 6, 7, 8, 9],
    #         'l2_leaf_reg': [1, 3, 5, 7, 9],
    #         'random_strength': [0.1, 0.5, 1.0, 2.0],
    #         'bagging_temperature': [0, 0.5, 1.0],
    #         'subsample': [0.7, 0.8, 0.9, 1.0],
    #         'task_type': 'GPU',
    #     },
        
    #     # Neural networks with tuning
    #     'NN_TORCH': {
    #         'num_layers': [2, 3, 4],
    #         'hidden_size': [128, 256, 512],
    #         'dropout_prob': [0.0, 0.1, 0.2, 0.3],
    #         'learning_rate': [1e-4, 1e-3, 1e-2],
    #         'num_epochs': [50, 100, 150],
    #         'activation': ['relu', 'elu', 'tanh', 'leaky_relu'],
    #         'use_batchnorm': [True, False],
    #     },
        
        # # Random Forest with tuning
        # 'RF': {
        #     'n_estimators': [100, 200, 300, 500],
        #     'max_depth': [10, 15, 20, None],
        #     'max_features': [0.5, 0.7, 0.9, 'auto', 'sqrt'],
        #     'min_samples_split': [2, 5, 10],
        #     'min_samples_leaf': [1, 2, 4],
        #     'bootstrap': [True, False],
        # },
        
        # # Extra Trees with tuning
        # 'XT': {
        #     'n_estimators': [100, 200, 300, 500],
        #     'max_depth': [10, 15, 20, None],
        #     'max_features': [0.5, 0.7, 0.9, 'auto', 'sqrt'],
        #     'min_samples_split': [2, 5, 10],
        #     'min_samples_leaf': [1, 2, 4],
        #     'bootstrap': [True, False],
        # },
        
        # # KNN with tuning
        # 'KNN': {
        #     'n_neighbors': [3, 5, 7, 10, 15, 20, 30, 50],
        #     'weights': ['uniform', 'distance'],
        #     'metric': ['euclidean', 'minkowski', 'manhattan'],
        # },
        
        # # Linear models with tuning
        # 'LR': {
        #     'fit_intercept': [True, False],
        #     'normalize': [True, False],
        #     'alpha': [0.0001, 0.001, 0.01, 0.1, 1.0, 10.0],
        # }
    # },
    verbosity=2,
    num_cpus=4,
    num_gpus=1
)

Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.4.0
Python Version:     3.11.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Sun Nov 10 10:07:59 UTC 2024
CPU Count:          4
Memory Avail:       29.58 GB / 31.35 GB (94.3%)
Disk Space Avail:   19.50 GB / 19.52 GB (99.9%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=3, num_bag_folds=5, num_bag_sets=3
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determine whether the input data is affected by stacked overfitting and enable or disable stacking as a consequence.
	This is used to identify the optimal `num_stack_levels` value. Copies of AutoGluon will be fit on subsets of the da

[1000]	valid_set's rmse: 0.0558599
[2000]	valid_set's rmse: 0.0558324
[1000]	valid_set's rmse: 0.0564653
[2000]	valid_set's rmse: 0.0564326
[1000]	valid_set's rmse: 0.0565349
[2000]	valid_set's rmse: 0.056487
[1000]	valid_set's rmse: 0.0561615
[2000]	valid_set's rmse: 0.056124
[1000]	valid_set's rmse: 0.056258
[2000]	valid_set's rmse: 0.0562204
[1000]	valid_set's rmse: 0.0563848
[2000]	valid_set's rmse: 0.0563447
[1000]	valid_set's rmse: 0.0559938
[2000]	valid_set's rmse: 0.0559543
[1000]	valid_set's rmse: 0.0561521
[2000]	valid_set's rmse: 0.0561355
[1000]	valid_set's rmse: 0.0564431
[2000]	valid_set's rmse: 0.0563999
[1000]	valid_set's rmse: 0.0563384
[2000]	valid_set's rmse: 0.0563159
[1000]	valid_set's rmse: 0.056555
[2000]	valid_set's rmse: 0.0565131
[1000]	valid_set's rmse: 0.056206
[2000]	valid_set's rmse: 0.0561808
[1000]	valid_set's rmse: 0.0560948
[2000]	valid_set's rmse: 0.0560562
[3000]	valid_set's rmse: 0.0560608
[1000]	valid_set's rmse: 0.0561655
[2000]	valid_set's rmse: 

	-0.0562	 = Validation score   (-root_mean_squared_error)
	601.16s	 = Training   runtime
	142.07s	 = Validation runtime
Fitting model: LightGBM_BAG_L1 ... Training model for up to 2545.44s of the 9141.30s of remaining time.
Specified total num_gpus: 1, but only 0 are available. Will use 0 instead
	Fitting 15 child models (S1F1 - S3F5) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=2, gpus=0)


[1000]	valid_set's rmse: 0.0556326
[1000]	valid_set's rmse: 0.0562041
[1000]	valid_set's rmse: 0.0558994
[1000]	valid_set's rmse: 0.0560507
[1000]	valid_set's rmse: 0.0561694
[1000]	valid_set's rmse: 0.0557555
[1000]	valid_set's rmse: 0.0561631
[1000]	valid_set's rmse: 0.056093
[1000]	valid_set's rmse: 0.0563617
[1000]	valid_set's rmse: 0.0558534
[1000]	valid_set's rmse: 0.0559526
[1000]	valid_set's rmse: 0.0559758


	-0.056	 = Validation score   (-root_mean_squared_error)
	271.87s	 = Training   runtime
	51.56s	 = Validation runtime
Fitting model: RandomForestMSE_BAG_L1 ... Training model for up to 2220.38s of the 8816.23s of remaining time.
Specified total num_gpus: 1, but only 0 are available. Will use 0 instead
	-0.0566	 = Validation score   (-root_mean_squared_error)
	411.39s	 = Training   runtime
	18.81s	 = Validation runtime
Fitting model: CatBoost_BAG_L1 ... Training model for up to 1788.62s of the 8384.47s of remaining time.
Specified total num_gpus: 1, but only 0 are available. Will use 0 instead
	Fitting 15 child models (S1F1 - S3F5) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=2, gpus=0)
	-0.056	 = Validation score   (-root_mean_squared_error)
	1347.99s	 = Training   runtime
	1.44s	 = Validation runtime
Fitting model: ExtraTreesMSE_BAG_L1 ... Training model for up to 438.31s of the 7034.16s of remaining time.
Specified total num_gpus: 1, but only 0 are available. W

...

	-0.0561	 = Validation score   (-root_mean_squared_error)
	749.93s	 = Training   runtime
	187.21s	 = Validation runtime
Fitting model: LightGBM_BAG_L1 ... Training model for up to 28507.28s of the 28507.27s of remaining time.
Specified total num_gpus: 1, but only 0 are available. Will use 0 instead
	Fitting 15 child models (S1F1 - S3F5) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=2, gpus=0)


[1000]	valid_set's rmse: 0.0561595
[1000]	valid_set's rmse: 0.0559825
[1000]	valid_set's rmse: 0.0559111
[1000]	valid_set's rmse: 0.0559614
[1000]	valid_set's rmse: 0.0561798
[1000]	valid_set's rmse: 0.0555203
[1000]	valid_set's rmse: 0.0559605
[1000]	valid_set's rmse: 0.0556618
[1000]	valid_set's rmse: 0.0562817
[1000]	valid_set's rmse: 0.0561644


	-0.056	 = Validation score   (-root_mean_squared_error)
	297.71s	 = Training   runtime
	57.0s	 = Validation runtime
Fitting model: RandomForestMSE_BAG_L1 ... Training model for up to 28150.85s of the 28150.84s of remaining time.
Specified total num_gpus: 1, but only 0 are available. Will use 0 instead
	-0.0565	 = Validation score   (-root_mean_squared_error)
	500.62s	 = Training   runtime
	20.78s	 = Validation runtime
Fitting model: CatBoost_BAG_L1 ... Training model for up to 27625.90s of the 27625.89s of remaining time.
Specified total num_gpus: 1, but only 0 are available. Will use 0 instead
	Fitting 15 child models (S1F1 - S3F5) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=2, gpus=0)
	-0.0559	 = Validation score   (-root_mean_squared_error)
	1541.61s	 = Training   runtime
	1.65s	 = Validation runtime
Fitting model: ExtraTreesMSE_BAG_L1 ... Training model for up to 26081.66s of the 26081.65s of remaining time.
Specified total num_gpus: 1, but only 0 are avail

[1000]	valid_set's rmse: 0.0561933


	Ran out of time, early stopping on iteration 1439. Best iteration is:
	[1439]	valid_set's rmse: 0.0561671
	Time limit exceeded... Skipping LightGBM_r131_BAG_L1.
Fitting model: NeuralNetFastAI_r191_BAG_L1 ... Training model for up to 647.75s of the 647.74s of remaining time.
Specified total num_gpus: 1, but only 0 are available. Will use 0 instead
	Fitting 15 child models (S1F1 - S3F5) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=2, gpus=0)
	Ran out of time, stopping training early. (Stopping on epoch 0)
	Ran out of time, stopping training early. (Stopping on epoch 0)
	Ran out of time, stopping training early. (Stopping on epoch 0)
	Ran out of time, stopping training early. (Stopping on epoch 0)
	Ran out of time, stopping training early. (Stopping on epoch 0)
	Ran out of time, stopping training early. (Stopping on epoch 0)
	Ran out of time, stopping training early. (Stopping on epoch 0)
	Ran out of time, stopping training early. (Stopping on epoch 0)
	Ran out of 

In [11]:
leaderboard = predictor.leaderboard()
leaderboard

,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L2,-0.055929,root_mean_squared_error,92.141220,3214.440040,0.009937,1.046002,2,True,14
1,CatBoost_BAG_L1,-0.055948,root_mean_squared_error,1.648799,1541.612821,1.648799,1541.612821,1,True,4
2,CatBoost_r177_BAG_L1,-0.055951,root_mean_squared_error,1.287883,1147.389678,1.287883,1147.389678,1,True,10
3,LightGBM_BAG_L1,-0.055953,root_mean_squared_error,57.000546,297.706278,57.000546,297.706278,1,True,2
4,XGBoost_BAG_L1,-0.055968,root_mean_squared_error,17.392119,310.859098,17.392119,310.859098,1,True,7
5,LightGBMLarge_BAG_L1,-0.055969,root_mean_squared_error,32.194056,226.685260,32.194056,226.685260,1,True,9
6,LightGBMXT_BAG_L1,-0.056149,root_mean_squared_error,187.206902,749.934230,187.206902,749.934230,1,True,1
7,NeuralNetFastAI_BAG_L1,-0.056432,root_mean_squared_error,12.508889,4270.660202,12.508889,4270.660202,1,True,6
8,ExtraTreesMSE_BAG_L1,-0.056447,root_mean_squared_error,20.520822,382.729223,20.520822,382.729223,1,True,5
9,NeuralNetTorch_r79_BAG_L1,-0.056465,root_mean_squared_error,6.668729,15357.566390,6.668729,15357.566390,1,True,11


In [12]:
models = leaderboard['model'].to_list()
oofs_dict = {}
for m in models:
    oofs_dict[m] = predictor.predict_oof(model=m)

oofs_df = pd.DataFrame(oofs_dict)

In [13]:
predictor2 = TabularPredictor.load('/kaggle/working/AutogluonModels/full_hpo')
test_preds = {}
for m in models:
    test_preds[m] = predictor2.predict(test_final, model=m)

test_preds_df = pd.DataFrame(test_preds)

In [14]:
# for col in oofs_df.columns:
#     oofs_df[col] = oofs_df[col] + train['y']
#     test_preds_df[col] = test_preds_df[col] + test['y']

In [15]:
samp = pd.read_csv('/kaggle/input/playground-series-s5e10/sample_submission.csv')
samp['accident_risk'] = test_preds_df.iloc[:,0]
samp.to_csv('autogluon_meta_8_models.csv', index=False)

In [16]:
leaderboard.to_csv('leaderboard_autogluon_residuals.csv', index=False)
oofs_df.to_csv('oofs_autogluon_residuals.csv', index=False)
test_preds_df.to_csv('test_preds_autogluon_residuals.csv', index=False)